# Week 4: Exercise 8 - SQLite Memory Store

**Goal:** Give your agent persistent memory across sessions.

No API key needed for this exercise.


## The Database: Three Tables, Fully Explained

`MemoryStore` creates **three tables**. Read this before implementing —
every method below is just SQL against one of these tables.

```
conversations  = the chat log   (append-only: every message ever saved)
user_facts     = the memory     (one CURRENT value per fact: name, language...)
skills         = the procedures (one CURRENT version per skill)
```

### Table 1: `conversations` — the chat log

| column | type | flags | meaning |
|---|---|---|---|
| `id` | INTEGER | **PRIMARY KEY** AUTOINCREMENT | row number, assigns itself 1, 2, 3... |
| `session_id` | TEXT | NOT NULL | which conversation this row belongs to |
| `role` | TEXT | NOT NULL | `'user'` or `'assistant'` (same roles as the OpenAI messages format) |
| `content` | TEXT | NOT NULL | the message text |
| `timestamp` | TEXT | DEFAULT `datetime('now')` | filled in automatically if omitted |

**Which column is the primary key?** `id` (the `pk=1` column in `PRAGMA table_info`).

**Why does this table need `id` at all?** Because nothing else in the row is
unique — the same person can say "Hello!" twice. A primary key must uniquely
identify each row, so we add a synthetic counter. This is called a
**surrogate key**: it has no meaning, it's just a row number.

**What AUTOINCREMENT does:** never put `id` in your INSERT — SQLite assigns
the next number itself. Bonus: `ORDER BY id` = insertion order = conversation
order (that's how `get_conversation` returns messages in the right sequence).

**Note on `session_id`:** it is *not* a foreign key — there is no `sessions`
table it points to. It's an informal grouping label you filter on with
`WHERE session_id = ?`. Production databases would formalize this; teaching
schemas keep it loose.


### Table 2: `user_facts` — key/value memory

| column | type | flags | meaning |
|---|---|---|---|
| `key` | TEXT | **PRIMARY KEY** | the fact's name: `'name'`, `'language'`... |
| `value` | TEXT | NOT NULL | the fact's current value |
| `category` | TEXT | DEFAULT `'preference'` | `'identity'`, `'preference'`, ... |
| `updated` | TEXT | DEFAULT `datetime('now')` | last-write time |

### Table 3: `skills` — saved procedures

| column | type | flags | meaning |
|---|---|---|---|
| `name` | TEXT | **PRIMARY KEY** | the skill's name: `'git_commit'`... |
| `content` | TEXT | NOT NULL | the skill's body (how to do it) |
| `description` | TEXT | DEFAULT `''` | short label |
| `created` | TEXT | DEFAULT `datetime('now')` | first-write time |

**These two use a different key style: no `id` column.** The fact's name IS
its identity — a **natural key**. The database enforces "one row per key":
`INSERT INTO user_facts (key, value) VALUES ('name', 'Bob')` when a row with
key `'name'` already exists fails with
`UNIQUE constraint failed: user_facts.key`.

**That failure is why `save_user_fact` uses an UPSERT:**

```sql
INSERT INTO user_facts (key, value, category) VALUES (?, ?, ?)
ON CONFLICT(key) DO UPDATE SET value = excluded.value,
                               category = excluded.category
```

`ON CONFLICT(key) DO UPDATE` = "if the key already exists, don't crash —
turn this INSERT into an UPDATE of that row". (`excluded` = "the row you
were trying to insert".) Same pattern for `save_skill` on `name`.


### The column flags, in one place

| flag | effect |
|---|---|
| `NOT NULL` | mandatory — omit the column in INSERT → `IntegrityError` |
| `DEFAULT <value>` | auto-fill when you omit the column (`'preference'`, `datetime('now')`, `''`) |
| `PRIMARY KEY` | unique row identity; implies UNIQUE + fast lookups by that column |

### The design lesson (this IS schema design)

| table | key style | write pattern | why |
|---|---|---|---|
| `conversations` | surrogate (`id`) | append-only, grows forever | history must never overwrite itself |
| `user_facts` | natural (`key`) | upsert | a fact has one *current* value |
| `skills` | natural (`name`) | upsert | a skill has one *current* version |

Choosing which key style fits the access pattern is the whole decision.

### Inspect any schema yourself

```python
# inside any notebook, once self.conn exists:
for row in self.conn.execute("PRAGMA table_info(conversations)"):
    print(row["cid"], row["name"], row["type"], row["notnull"], row["dflt_value"], row["pk"])
# pk=1 marks the primary-key column
```


## Step 1: Implement MemoryStore

**TODO:** Create three tables: `conversations`, `user_facts`, `skills`.


In [ ]:
import sqlite3, json
from datetime import datetime
from pathlib import Path

class MemoryStore:
    """TODO: Implement SQLite memory store."""

    def __init__(self, db_path="agent_memory.db"):
        self.db_path = Path(db_path)
        self.conn = sqlite3.connect(str(self.db_path))
        self.conn.row_factory = sqlite3.Row
        self._init_db()

    def _init_db(self):
        """TODO: Create tables: conversations, user_facts, skills."""
        # Hint: self.conn.executescript with CREATE TABLE IF NOT EXISTS ...
        #       conversations(id, session_id, role, content, timestamp)
        #       user_facts(key, value, category, updated)
        #       skills(name, content, description, created)
        #       then self.conn.commit()
        self.conn.executescript("""
            CREATE TABLE IF NOT EXISTS conversations (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT NOT NULL,
                role TEXT NOT NULL,
                content TEXT NOT NULL,
                timestamp TEXT DEFAULT (datetime('now'))
            );
            CREATE TABLE IF NOT EXISTS user_facts (
                key TEXT PRIMARY KEY,
                value TEXT NOT NULL,
                category TEXT DEFAULT 'preference',
                updated TEXT DEFAULT (datetime('now'))
            );
            CREATE TABLE IF NOT EXISTS skills (
                name TEXT PRIMARY KEY,
                content TEXT NOT NULL,
                description TEXT DEFAULT '',
                created TEXT DEFAULT (datetime('now'))
            );
        """)
        self.conn.commit()

    def save_message(self, session_id, role, content):
        """TODO: Save a message to conversation history."""
        # Hint: INSERT INTO conversations (session_id, role, content) VALUES (?, ?, ?)
        self.conn.execute(
            "INSERT INTO conversations (session_id, role, content) VALUES (?, ?, ?)",
            (session_id, role, content))
        self.conn.commit()

    def get_conversation(self, session_id, limit=100):
        """TODO: Get conversation history."""
        # Hint: SELECT role, content FROM conversations WHERE session_id = ?
        #       ORDER BY id LIMIT ?   -> fetchall, return list of dicts
        rows = self.conn.execute(
            "SELECT role, content FROM conversations WHERE session_id = ? ORDER BY id LIMIT ?",
            (session_id, limit)).fetchall()
        return [{"role": r["role"], "content": r["content"]} for r in rows]

    def save_user_fact(self, key, value, category="preference"):
        """TODO: Save a user preference/fact."""
        # Hint: INSERT INTO user_facts ... ON CONFLICT(key) DO UPDATE SET value = excluded.value
        self.conn.execute("""
            INSERT INTO user_facts (key, value, category)
            VALUES (?, ?, ?)
            ON CONFLICT(key) DO UPDATE SET
                value = excluded.value,
                category = excluded.category,
                updated = datetime('now')
        """, (key, value, category))
        self.conn.commit()

    def get_user_fact(self, key):
        """TODO: Get a user fact by key."""
        # Hint: SELECT value FROM user_facts WHERE key = ?   -> fetchone
        row = self.conn.execute(
            "SELECT value FROM user_facts WHERE key = ?", (key,)).fetchone()
        return row["value"] if row else None

    def save_skill(self, name, content, description=""):
        """TODO: Save a skill/procedure."""
        # Hint: INSERT INTO skills ... ON CONFLICT(name) DO UPDATE SET content = excluded.content
        self.conn.execute("""
            INSERT INTO skills (name, content, description)
            VALUES (?, ?, ?)
            ON CONFLICT(name) DO UPDATE SET
                content = excluded.content,
                description = excluded.description
        """, (name, content, description))
        self.conn.commit()

    def get_skill(self, name):
        """TODO: Get a skill."""
        # Hint: SELECT content FROM skills WHERE name = ?   -> fetchone
        row = self.conn.execute(
            "SELECT content FROM skills WHERE name = ?", (name,)).fetchone()
        return row["content"] if row else None

    def get_stats(self):
        """TODO: Return statistics."""
        # Hint: {"conversations": COUNT(*), "facts": COUNT(*), "skills": COUNT(*)}
        def count(table):
            return self.conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        return {"conversations": count("conversations"),
                "facts": count("user_facts"),
                "skills": count("skills")}


## Step 2: Test It


In [ ]:
mem = MemoryStore("test_mem.db")

mem.save_message("s1", "user", "Hello!")
mem.save_message("s1", "assistant", "Hi there!")

mem.save_user_fact("name", "Alex", "identity")
mem.save_user_fact("language", "Python", "preference")

mem.save_skill("git_commit", "git add -A && git commit -m", "Git workflow")

print("Messages:", mem.get_conversation("s1"))
print("Name:", mem.get_user_fact("name"))
print("Skill:", mem.get_skill("git_commit"))
print("Stats:", mem.get_stats())

mem.conn.close()
import os
os.remove("test_mem.db")
print("Cleaned up!")


Messages: [{'role': 'user', 'content': 'Hello!'}, {'role': 'assistant', 'content': 'Hi there!'}]
Name: Alex
Skill: git add -A && git commit -m
Stats: {'conversations': 2, 'facts': 2, 'skills': 1}
Cleaned up!


## Key Takeaways
- SQLite is built into Python
- `ON CONFLICT ... DO UPDATE` = upsert
